# Weekend 2 — Train the U-Net

Train on BBBC038/DSB2018, log to W&B, and check Dice/IoU on a held-out split before wiring it into the app.

In [ ]:
import sys
sys.path.append("../train")

from train_unet import train

In [ ]:
checkpoint_path, info = train(config_path="../train/config.yaml", wandb_mode="offline")
print(checkpoint_path, info["best_val_dice"])

In [ ]:
import sys
sys.path.append("../app")

from segment.unet_model import run_unet
from dataset import DSBNucleiDataset
import numpy as np
import matplotlib.pyplot as plt

ds = DSBNucleiDataset("../data/BBBC038/stage1_train", image_size=256)
image_tensor, mask_tensor = ds[5]
image = (image_tensor.permute(1, 2, 0).numpy() * 255).astype(np.uint8)

pred_mask = run_unet(image, weights_path="../models/unet_v1.pt")

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(image)
axes[0].set_title("input")
axes[1].imshow(mask_tensor[0], cmap="gray")
axes[1].set_title("ground truth")
axes[2].imshow(pred_mask > 0, cmap="gray")
axes[2].set_title(f"U-Net prediction ({pred_mask.max()} instances)")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()